# 🚀 VWAP Mean Reversion Trading Strategy
**Complete Trading Framework with Delta Exchange Integration**

## 📋 Blocks Overview:
1. **API Credentials & Authentication** - Delta Exchange Setup
2. **Trading Parameters** - Strategy Configuration
3. **Technical Indicators** - TA-Lib & Strategy Logic
4. **Data Fetching** - Delta Exchange Historical Data
5. **Backtest Engine** - Backtrader Execution & Results
6. **Trade Logs** - TradingView Format Reports
7. **Interactive Charts** - Candlestick + Indicators (Scrollable)

**Run blocks sequentially from top to bottom ↓**

---
## BLOCK 1️⃣ : API CREDENTIALS & DELTA EXCHANGE AUTHENTICATION

In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# BLOCK 1: API CREDENTIALS & DELTA EXCHANGE AUTHENTICATION
# ═══════════════════════════════════════════════════════════════════════════════

import hashlib
import hmac
import requests
import time
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

# ═════════════════════════════════════════════════════════════════════════════════
# 🔐 DELTA EXCHANGE API CREDENTIALS
# ═════════════════════════════════════════════════════════════════════════════════

API_KEY = 'enter api key '
API_SECRET = 'enter api secret'
BASE_URL = 'https://api.delta.exchange'
HISTORY_URL = 'https://api.india.delta.exchange/v2/history/candles'

# ═════════════════════════════════════════════════════════════════════════════════
# ✅ DELTA EXCHANGE API VERIFICATION
# ═════════════════════════════════════════════════════════════════════════════════

def generate_signature(secret, message):
    """Generate HMAC-SHA256 signature for API authentication"""
    message = bytes(message, 'utf-8')
    secret = bytes(secret, 'utf-8')
    hash_obj = hmac.new(secret, message, hashlib.sha256)
    return hash_obj.hexdigest()

# Verify API credentials format
test_message = "test_authentication"
test_signature = generate_signature(API_SECRET, test_message)

print("╔" + "═"*80 + "╗")
print("║" + "✅ DELTA EXCHANGE API CREDENTIALS VERIFICATION".center(80) + "║")
print("╚" + "═"*80 + "╝\n")

print(f"🔐 API Key:        {API_KEY[:15]}...{API_KEY[-5:]}")
print(f"🔑 API Secret:     {API_SECRET[:10]}...{API_SECRET[-5:]}")
print(f"🌐 Base URL:       {BASE_URL}")
print(f"📡 History URL:    {HISTORY_URL}")
print(f"\n✓ Signature Algorithm: HMAC-SHA256")
print(f"✓ Test Signature:  {test_signature[:20]}...")
print(f"\n⏰ Current Time UTC: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')}")
print(f"✅ Authentication Ready!\n")

╔════════════════════════════════════════════════════════════════════════════════╗
║                 ✅ DELTA EXCHANGE API CREDENTIALS VERIFICATION                  ║
╚════════════════════════════════════════════════════════════════════════════════╝

🔐 API Key:        enter api key ... key 
🔑 API Secret:     enter api ...ecret
🌐 Base URL:       https://api.delta.exchange
📡 History URL:    https://api.india.delta.exchange/v2/history/candles

✓ Signature Algorithm: HMAC-SHA256
✓ Test Signature:  92f212728f775ec38e68...

⏰ Current Time UTC: 2026-05-11 05:34:46
✅ Authentication Ready!



---
## BLOCK 2️⃣ : TRADING PARAMETERS & STRATEGY CONFIGURATION

In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# BLOCK 2: TRADING PARAMETERS & STRATEGY CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("⚙️  TRADING STRATEGY PARAMETERS".center(80))
print("="*80 + "\n")

# ═════════════════════════════════════════════════════════════════════════════════
# 🎯 TRADING MODE SELECTION
# ═════════════════════════════════════════════════════════════════════════════════

TRADING_MODE = "BOTH"  # Options: "LONG_ONLY", "SHORT_ONLY", "BOTH"

# ═════════════════════════════════════════════════════════════════════════════════
# 📊 VWAP INDICATOR SETTINGS
# ═════════════════════════════════════════════════════════════════════════════════

VWAP_ANCHOR = "Session"              # Session / Week / Month
VWAP_STD_DEV_MULTIPLIER = 1.8        # Standard deviation bands (1.5-2.5)

# ═════════════════════════════════════════════════════════════════════════════════
# 📍 ENTRY PARAMETERS
# ═════════════════════════════════════════════════════════════════════════════════

# Set LONG_ENABLED and SHORT_ENABLED based on TRADING_MODE
if TRADING_MODE == "LONG_ONLY":
    LONG_ENABLED = True
    SHORT_ENABLED = False
elif TRADING_MODE == "SHORT_ONLY":
    LONG_ENABLED = False
    SHORT_ENABLED = True
else:  # BOTH
    LONG_ENABLED = True
    SHORT_ENABLED = True

LONG_DISTANCE_PERCENT = 1         # Price > 1% below VWAP
SHORT_DISTANCE_PERCENT = 1         # Price > 0.5% above VWAP

# ═════════════════════════════════════════════════════════════════════════════════
# 🚪 EXIT PARAMETERS
# ═════════════════════════════════════════════════════════════════════════════════

USE_CENTER_VWAP_EXIT = True          # Exit at center VWAP
USE_DAY_END_EXIT = False             # Exit at day end
USE_TAKE_PROFIT = False              # Enable take profit
TAKE_PROFIT_PERCENT = 0.5            # 0.5% profit target
USE_STOP_LOSS = True                 # Enable stop loss
STOP_LOSS_PERCENT = 1.5              # 1.5% stop loss

# ═════════════════════════════════════════════════════════════════════════════════
# ⏰ TRADING SESSION (UTC - For consistency with API)
# ═════════════════════════════════════════════════════════════════════════════════

SESSION_START_HOUR = 8               # 08:00 UTC
SESSION_START_MIN = 0
SESSION_END_HOUR = 22
SESSION_END_MIN = 30                 # 22:30 UTC
SESSION_START = SESSION_START_HOUR * 60 + SESSION_START_MIN
SESSION_END = SESSION_END_HOUR * 60 + SESSION_END_MIN

# ═════════════════════════════════════════════════════════════════════════════════
# 💰 POSITION SIZING & RISK MANAGEMENT
# ═════════════════════════════════════════════════════════════════════════════════

INITIAL_CAPITAL = 100000             # Starting capital in USD
POSITION_SIZE = 1.0                  # BTC per trade
COMMISSION = 0.00                    # 0.05% per trade
SLIPPAGE_POINTS = 0                  # $0 slippage per entry/exit

# ═════════════════════════════════════════════════════════════════════════════════
# 📅 DATA PARAMETERS
# ═════════════════════════════════════════════════════════════════════════════════

BACKTEST_DAYS = 30                   # 30 days
TIMEFRAME = "1min"                   # 1-minute candles
SYMBOL = "BTCUSD"                    # BTC/USD perpetuals
DATA_CACHE_FILE = 'backtest_data.pkl'

# ═════════════════════════════════════════════════════════════════════════════════
# 📊 TECHNICAL INDICATOR PARAMETERS (Only Essential)
# ═════════════════════════════════════════════════════════════════════════════════

RSI_PERIOD = 14                      # RSI period for confirmation only
RSI_OVERBOUGHT = 65                  # Above this = Overbought
RSI_OVERSOLD = 35                    # Below this = Oversold

# ═════════════════════════════════════════════════════════════════════════════════
# 📋 DISPLAY PARAMETERS
# ═════════════════════════════════════════════════════════════════════════════════

params_display = f"""
🎯 TRADING MODE:
   Mode:                {TRADING_MODE}

📊 VWAP SETTINGS:
   Anchor:              {VWAP_ANCHOR}
   Std Dev Multiplier:  {VWAP_STD_DEV_MULTIPLIER}

📍 ENTRY CONDITIONS:
   LONG Enabled:        {LONG_ENABLED} (Price > {LONG_DISTANCE_PERCENT}% below VWAP)
   SHORT Enabled:       {SHORT_ENABLED} (Price > {SHORT_DISTANCE_PERCENT}% above VWAP)

🚪 EXIT CONDITIONS:
   VWAP Center Exit:    {USE_CENTER_VWAP_EXIT}
   Take Profit:         {USE_TAKE_PROFIT} ({TAKE_PROFIT_PERCENT}%)
   Stop Loss:           {USE_STOP_LOSS} ({STOP_LOSS_PERCENT}%)

⏰ TRADING SESSION (UTC):
   {SESSION_START_HOUR:02d}:{SESSION_START_MIN:02d} - {SESSION_END_HOUR:02d}:{SESSION_END_MIN:02d}

💰 POSITION SIZING:
   Initial Capital:     ${INITIAL_CAPITAL:,}
   Position Size:       {POSITION_SIZE} BTC
   Commission:          {COMMISSION}%

📅 DATA:
   Timeframe:           {TIMEFRAME}
   Symbol:              {SYMBOL}
   Backtest Days:       {BACKTEST_DAYS}
   
🎯 RSI CONFIRMATION:
   Period:              {RSI_PERIOD}
   Overbought:          {RSI_OVERBOUGHT}
   Oversold:            {RSI_OVERSOLD}
"""

print(params_display)
print("✅ Parameters configured successfully!\n")


                        ⚙️  TRADING STRATEGY PARAMETERS                         


🎯 TRADING MODE:
   Mode:                BOTH

📊 VWAP SETTINGS:
   Anchor:              Session
   Std Dev Multiplier:  1.8

📍 ENTRY CONDITIONS:
   LONG Enabled:        True (Price > 1% below VWAP)
   SHORT Enabled:       True (Price > 1% above VWAP)

🚪 EXIT CONDITIONS:
   VWAP Center Exit:    True
   Take Profit:         False (0.5%)
   Stop Loss:           True (1.5%)

⏰ TRADING SESSION (UTC):
   08:00 - 22:30

💰 POSITION SIZING:
   Initial Capital:     $100,000
   Position Size:       1.0 BTC
   Commission:          0.0%

📅 DATA:
   Timeframe:           1min
   Symbol:              BTCUSD
   Backtest Days:       30

🎯 RSI CONFIRMATION:
   Period:              14
   Overbought:          65
   Oversold:            35

✅ Parameters configured successfully!



---
## BLOCK 3️⃣ : TECHNICAL INDICATORS & STRATEGY LOGIC

In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# BLOCK 3: TECHNICAL INDICATORS & STRATEGY LOGIC
# ═══════════════════════════════════════════════════════════════════════════════

try:
    import talib
    TALIB_AVAILABLE = True
    print("✅ TA-Lib library imported successfully")
except ImportError:
    TALIB_AVAILABLE = False
    print("⚠️  TA-Lib not available, using NumPy alternatives")

print(f"\n" + "="*80)
print("📊 TECHNICAL INDICATORS - ESSENTIAL ONLY".center(80))
print("="*80 + "\n")

# ═════════════════════════════════════════════════════════════════════════════════
# VWAP CALCULATION (PRIMARY INDICATOR)
# ═════════════════════════════════════════════════════════════════════════════════

def calculate_vwap_bands(df, anchor, multiplier):
    """
    Calculate VWAP (Volume Weighted Average Price) and confidence bands
    This is the PRIMARY trading indicator
    """
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['tp'] = (df['high'] + df['low'] + df['close']) / 3  # Typical Price
    
    # Group by anchor period
    if anchor == "Session":
        df['period'] = df['timestamp'].dt.date
    elif anchor == "Week":
        df['period'] = df['timestamp'].dt.to_period('W')
    else:  # Month
        df['period'] = df['timestamp'].dt.to_period('M')
    
    # Calculate VWAP
    df['tp_volume'] = df['tp'] * df['volume']
    df['cum_tp_volume'] = df.groupby('period')['tp_volume'].cumsum()
    df['cum_volume'] = df.groupby('period')['volume'].cumsum()
    df['vwap'] = df['cum_tp_volume'] / df['cum_volume']
    
    # Calculate Standard Deviation for confidence bands
    df['tp_sq_volume'] = (df['tp'] ** 2) * df['volume']
    df['cum_tp_sq_volume'] = df.groupby('period')['tp_sq_volume'].cumsum()
    variance = (df['cum_tp_sq_volume'] / df['cum_volume']) - (df['vwap'] ** 2)
    df['stdev'] = np.sqrt(variance.clip(lower=0))
    
    # Upper and Lower bands for reversion strategy
    df['upper_band'] = df['vwap'] + df['stdev'] * multiplier
    df['lower_band'] = df['vwap'] - df['stdev'] * multiplier
    
    return df

# ═════════════════════════════════════════════════════════════════════════════════
# RSI CALCULATION (CONFIRMATION ONLY)
# ═════════════════════════════════════════════════════════════════════════════════

def calculate_rsi(df, period=14):
    """
    Calculate Relative Strength Index (RSI)
    Used for CONFIRMATION only, not primary signal
    """
    delta = df['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    df['rsi'] = 100 - (100 / (1 + rs))
    return df

# ═════════════════════════════════════════════════════════════════════════════════
# STRATEGY ENTRY/EXIT LOGIC
# ═════════════════════════════════════════════════════════════════════════════════

def check_session_time(timestamp, start_min, end_min):
    """Check if within trading session (UTC)"""
    minutes_in_day = timestamp.hour * 60 + timestamp.minute
    return start_min <= minutes_in_day <= end_min

def check_long_entry(close, vwap, lower_band, distance_pct, rsi=None):
    """
    Check LONG entry conditions
    Primary: Price at lower band + outside distance threshold
    Confirmation: RSI oversold
    """
    below_lower_band = close < lower_band
    outside_distance = close < (vwap * (1 - distance_pct / 100))
    
    if rsi is not None:
        rsi_oversold = rsi < RSI_OVERSOLD
        return below_lower_band and outside_distance and rsi_oversold
    
    return below_lower_band and outside_distance

def check_short_entry(close, vwap, upper_band, distance_pct, rsi=None):
    """
    Check SHORT entry conditions
    Primary: Price at upper band + outside distance threshold
    Confirmation: RSI overbought
    """
    above_upper_band = close > upper_band
    outside_distance = close > (vwap * (1 + distance_pct / 100))
    
    if rsi is not None:
        rsi_overbought = rsi > RSI_OVERBOUGHT
        return above_upper_band and outside_distance and rsi_overbought
    
    return above_upper_band and outside_distance

print("✓ VWAP & Confidence Bands (PRIMARY)")
print("✓ RSI (14-period, CONFIRMATION ONLY)")
print("✓ Volume Analysis (for VWAP calculation)")
print("\n✅ Essential indicators configured!\n")

✅ TA-Lib library imported successfully

                    📊 TECHNICAL INDICATORS - ESSENTIAL ONLY                     

✓ VWAP & Confidence Bands (PRIMARY)
✓ RSI (14-period, CONFIRMATION ONLY)
✓ Volume Analysis (for VWAP calculation)

✅ Essential indicators configured!



---
## BLOCK 4️⃣ : DATA FETCHING FROM DELTA EXCHANGE

In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
# BLOCK 4: DATA FETCHING FROM DELTA EXCHANGE (OPTIMIZED)
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("📥 FETCHING HISTORICAL OHLC DATA FROM DELTA EXCHANGE".center(80))
print("="*80 + "\n")

backtest_df = None

# Check cache first
if os.path.exists(DATA_CACHE_FILE):
    print(f"📂 Loading cached data from: {DATA_CACHE_FILE}\n")
    try:
        with open(DATA_CACHE_FILE, 'rb') as f:
            backtest_df = pickle.load(f)
        print(f"✓ Data loaded from cache")
        print(f"  Candles: {len(backtest_df):,}")
        print(f"  Date Range: {backtest_df['timestamp'].min()} to {backtest_df['timestamp'].max()}")
        print(f"  Price Range: ${backtest_df['low'].min():,.2f} - ${backtest_df['high'].max():,.2f}")
        print(f"  Avg Volume: {backtest_df['volume'].mean():,.4f} BTC")
        print(f"  Total Volume: {backtest_df['volume'].sum():,.2f} BTC\n")
    except Exception as e:
        print(f"⚠️  Cache load failed: {e}")
        print(f"Fetching fresh data...\n")
        backtest_df = None

if backtest_df is None:
    print(f"⏳ Fetching {BACKTEST_DAYS} days of {TIMEFRAME} OHLC data from Delta Exchange...\n")
    
    try:
        current_time = int(time.time())
        start_time = current_time - (BACKTEST_DAYS * 24 * 60 * 60)
        
        print(f"📅 Time Range: {datetime.fromtimestamp(start_time, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')} to {datetime.fromtimestamp(current_time, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
        print(f"📊 Expected Candles: {BACKTEST_DAYS * 24 * 60:,} (1-minute)")
        print(f"🔄 Fetching in batches (max 2000 candles per request)...\n")
        
        all_candles = []
        end_timestamp = current_time
        request_count = 0
        max_requests = 500
        batch_size = 2000 * 60  # 2000 candles
        
        while end_timestamp > start_time and request_count < max_requests:
            request_count += 1
            batch_start = max(end_timestamp - batch_size, start_time)
            
            params = {
                'resolution': '1m',
                'symbol': SYMBOL,
                'start': batch_start,
                'end': end_timestamp
            }
            
            try:
                response = requests.get(HISTORY_URL, params=params, timeout=15)
                
                if response.status_code == 200:
                    data = response.json()
                    
                    if data.get('success') and data.get('result'):
                        candles = data['result']
                        all_candles.extend(candles)
                        oldest_time = candles[-1]['time']
                        progress = min(100, ((current_time - oldest_time) / (current_time - start_time)) * 100)
                        
                        if request_count % 10 == 0 or request_count == 1:
                            print(f"  Request {request_count:3d}: ✓ {len(candles):4d} candles | Total: {len(all_candles):7,d} | Progress: {progress:5.1f}%")
                        
                        end_timestamp = oldest_time - 60
                        time.sleep(0.2)  # Rate limiting
                    else:
                        print(f"  Request {request_count}: ✗ No data returned")
                        break
                else:
                    print(f"  Request {request_count}: ✗ Status {response.status_code}")
                    break
            
            except requests.exceptions.Timeout:
                print(f"  Request {request_count}: ✗ Timeout")
                break
            except requests.exceptions.RequestException as e:
                print(f"  Request {request_count}: ✗ Error: {str(e)[:50]}")
                break
        
        print(f"\n✓ Fetch Complete: {request_count} API requests | {len(all_candles):,} candles\n")
        
        if len(all_candles) > 1000:
            # Convert to DataFrame
            backtest_df = pd.DataFrame(all_candles)
            backtest_df['timestamp'] = pd.to_datetime(backtest_df['time'], unit='s', utc=True)
            
            # Ensure correct OHLC column names
            if 'o' in backtest_df.columns:
                backtest_df.rename(columns={'o': 'open', 'h': 'high', 'l': 'low', 'c': 'close', 'v': 'volume'}, inplace=True)
            
            # Keep only essential columns
            backtest_df = backtest_df[['timestamp', 'open', 'high', 'low', 'close', 'volume']].copy()
            backtest_df = backtest_df.sort_values('timestamp').reset_index(drop=True)
            
            # Clean data: remove duplicates and ensure data quality
            backtest_df = backtest_df.drop_duplicates(subset=['timestamp']).reset_index(drop=True)
            
            # Save to cache
            with open(DATA_CACHE_FILE, 'wb') as f:
                pickle.dump(backtest_df, f)
            print(f"💾 Data cached to: {DATA_CACHE_FILE}\n")
        else:
            print(f"⚠️  Insufficient data: {len(all_candles)} candles (need > 1000)")
            backtest_df = None
    
    except Exception as e:
        print(f"✗ Error fetching data: {str(e)}")
        backtest_df = None

# Display data summary
if backtest_df is not None:
    print("╔" + "═"*78 + "╗")
    print("║" + "✅ DATA READY FOR BACKTESTING".center(78) + "║")
    print("╚" + "═"*78 + "╝")
    print(f"\n📊 DATA SUMMARY:")
    print(f"   Total Candles:       {len(backtest_df):>12,}")
    print(f"   Time Range:          {backtest_df['timestamp'].min().strftime('%Y-%m-%d %H:%M')} to {backtest_df['timestamp'].max().strftime('%Y-%m-%d %H:%M')} UTC")
    print(f"   Duration:            {(backtest_df['timestamp'].max() - backtest_df['timestamp'].min()).days} days")
    print(f"   Price Range:         ${backtest_df['low'].min():>12,.2f} - ${backtest_df['high'].max():>12,.2f}")
    print(f"   Avg Price:           ${backtest_df['close'].mean():>12,.2f}")
    print(f"   Avg Volume/Min:      {backtest_df['volume'].mean():>12,.4f} BTC")
    print(f"   Total Volume:        {backtest_df['volume'].sum():>12,.2f} BTC\n")
else:
    print("❌ Failed to load data. Check API credentials and internet connection.")


              📥 FETCHING HISTORICAL OHLC DATA FROM DELTA EXCHANGE               

📂 Loading cached data from: backtest_data.pkl

✓ Data loaded from cache
  Candles: 259,199
  Date Range: 2025-09-27 11:03:00+00:00 to 2026-03-26 11:01:00+00:00
  Price Range: $59,835.50 - $126,205.50
  Avg Volume: 7,866.2559 BTC
  Total Volume: 2,038,925,653.00 BTC

╔══════════════════════════════════════════════════════════════════════════════╗
║                         ✅ DATA READY FOR BACKTESTING                         ║
╚══════════════════════════════════════════════════════════════════════════════╝

📊 DATA SUMMARY:
   Total Candles:            259,199
   Time Range:          2025-09-27 11:03 to 2026-03-26 11:01 UTC
   Duration:            179 days
   Price Range:         $   59,835.50 - $  126,205.50
   Avg Price:           $   89,549.40
   Avg Volume/Min:        7,866.2559 BTC
   Total Volume:        2,038,925,653.00 BTC



---
## BLOCK 5️⃣ : BACKTEST ENGINE WITH BACKTRADER

In [5]:
# ═══════════════════════════════════════════════════════════════════════════════
# BLOCK 5: OPTIMIZED BACKTEST ENGINE WITH BACKTRADER
# ═══════════════════════════════════════════════════════════════════════════════

import backtrader as bt

print("\n" + "="*80)
print("🚀 RUNNING OPTIMIZED VWAP MEAN REVERSION BACKTEST".center(80))
print("="*80 + "\n")

# Global list to store trades for visualization
all_trades = []

# ═════════════════════════════════════════════════════════════════════════════════
# PROFESSIONAL VWAP MEAN REVERSION STRATEGY
# ═════════════════════════════════════════════════════════════════════════════════

class VWAPMeanReversionStrategy(bt.Strategy):
    """
    Professional VWAP Mean Reversion Strategy
    - Entry: Price at VWAP bands with RSI confirmation
    - Exit: VWAP center, Take Profit, Stop Loss with clean tracking
    """
    
    params = (
        ('vwap_stdev', VWAP_STD_DEV_MULTIPLIER),
        ('long_enabled', LONG_ENABLED),
        ('short_enabled', SHORT_ENABLED),
        ('long_distance', LONG_DISTANCE_PERCENT),
        ('short_distance', SHORT_DISTANCE_PERCENT),
        ('use_vwap_exit', USE_CENTER_VWAP_EXIT),
        ('use_tp', USE_TAKE_PROFIT),
        ('tp_percent', TAKE_PROFIT_PERCENT),
        ('use_sl', USE_STOP_LOSS),
        ('sl_percent', STOP_LOSS_PERCENT),
        ('session_start', SESSION_START),
        ('session_end', SESSION_END),
        ('rsi_period', RSI_PERIOD),
    )
    
    def __init__(self):
        # VWAP using typical price
        tp = (self.data.high + self.data.low + self.data.close) / 3.0
        self.vwap = bt.indicators.SimpleMovingAverage(tp, period=20)
        
        # RSI for confirmation
        self.rsi = bt.indicators.RSI(self.data.close, period=self.p.rsi_period)
        
        # Standard deviation for bands
        self.stdev = bt.indicators.StdDev(self.data.close, period=20)
        
        # Trade tracking
        self.entry_price = None
        self.entry_time = None
        self.entry_type = None
        self.trade_count = 0
        self.in_position = False
        
    def is_trading_session(self):
        """Check if within active trading session (UTC)"""
        current_time = self.data.datetime.datetime(0)
        minutes_in_day = current_time.hour * 60 + current_time.minute
        return self.p.session_start <= minutes_in_day <= self.p.session_end
    
    def next(self):
        # Skip if not enough data
        if len(self) < 50:
            return
        
        # Exit if outside trading session
        if not self.is_trading_session():
            if self.position:
                self.close()
            return
        
        current_price = self.data.close[0]
        current_time = self.data.datetime.datetime(0)
        current_volume = self.data.volume[0]
        vwap = self.vwap[0]
        stdev = self.stdev[0]
        rsi = self.rsi[0]
        
        upper_band = vwap + (stdev * self.p.vwap_stdev)
        lower_band = vwap - (stdev * self.p.vwap_stdev)
        
        # ═════════════════════════════════════════════════════════════════════════════
        # EXIT LOGIC
        # ═════════════════════════════════════════════════════════════════════════════
        
        if self.position:
            exit_signal = False
            exit_reason = None
            exit_price = current_price
            
            if self.entry_type == 'LONG':
                # Exit at VWAP center
                if self.p.use_vwap_exit and current_price >= vwap:
                    exit_signal = True
                    exit_reason = "VWAP Exit"
                
                # Take Profit
                elif self.p.use_tp and self.entry_price:
                    tp_price = self.entry_price * (1 + self.p.tp_percent / 100)
                    if current_price >= tp_price:
                        exit_signal = True
                        exit_reason = "Take Profit"
                        exit_price = tp_price
                
                # Stop Loss
                elif self.p.use_sl and self.entry_price:
                    sl_price = self.entry_price * (1 - self.p.sl_percent / 100)
                    if current_price <= sl_price:
                        exit_signal = True
                        exit_reason = "Stop Loss"
                        exit_price = sl_price
            
            elif self.entry_type == 'SHORT':
                # Exit at VWAP center
                if self.p.use_vwap_exit and current_price <= vwap:
                    exit_signal = True
                    exit_reason = "VWAP Exit"
                
                # Take Profit
                elif self.p.use_tp and self.entry_price:
                    tp_price = self.entry_price * (1 - self.p.tp_percent / 100)
                    if current_price <= tp_price:
                        exit_signal = True
                        exit_reason = "Take Profit"
                        exit_price = tp_price
                
                # Stop Loss
                elif self.p.use_sl and self.entry_price:
                    sl_price = self.entry_price * (1 + self.p.sl_percent / 100)
                    if current_price >= sl_price:
                        exit_signal = True
                        exit_reason = "Stop Loss"
                        exit_price = sl_price
            
            if exit_signal:
                # Calculate P&L
                if self.entry_type == 'LONG':
                    pnl = (exit_price - self.entry_price) * POSITION_SIZE
                    pnl_pct = ((exit_price - self.entry_price) / self.entry_price) * 100
                else:  # SHORT
                    pnl = (self.entry_price - exit_price) * POSITION_SIZE
                    pnl_pct = ((self.entry_price - exit_price) / self.entry_price) * 100
                
                # Store trade for visualization
                trade_record = {
                    'trade_num': self.trade_count,
                    'entry_time': self.entry_time,
                    'entry_price': self.entry_price,
                    'entry_type': self.entry_type,
                    'entry_vwap': self.entry_vwap,
                    'entry_upper': self.entry_upper_band,
                    'entry_lower': self.entry_lower_band,
                    'entry_rsi': self.entry_rsi,
                    'exit_time': current_time,
                    'exit_price': exit_price,
                    'exit_vwap': vwap,
                    'exit_rsi': rsi,
                    'exit_reason': exit_reason,
                    'position_size': POSITION_SIZE,
                    'pnl': pnl,
                    'pnl_pct': pnl_pct,
                    'duration_bars': len(self) - self.entry_bar,
                }
                all_trades.append(trade_record)
                
                self.close()
                self.in_position = False
                return
        
        # ═════════════════════════════════════════════════════════════════════════════
        # ENTRY LOGIC
        # ═════════════════════════════════════════════════════════════════════════════
        
        if not self.position and not self.in_position:
            
            # LONG ENTRY
            if self.p.long_enabled:
                below_lower = current_price < lower_band
                outside_distance = current_price < (vwap * (1 - self.p.long_distance / 100))
                rsi_oversold = rsi < RSI_OVERSOLD
                
                if below_lower and outside_distance and rsi_oversold:
                    self.buy(size=POSITION_SIZE)
                    self.entry_type = 'LONG'
                    self.entry_price = current_price
                    self.entry_time = current_time
                    self.entry_vwap = vwap
                    self.entry_upper_band = upper_band
                    self.entry_lower_band = lower_band
                    self.entry_rsi = rsi
                    self.entry_bar = len(self)
                    self.trade_count += 1
                    self.in_position = True
                    return
            
            # SHORT ENTRY
            if self.p.short_enabled:
                above_upper = current_price > upper_band
                outside_distance = current_price > (vwap * (1 + self.p.short_distance / 100))
                rsi_overbought = rsi > RSI_OVERBOUGHT
                
                if above_upper and outside_distance and rsi_overbought:
                    self.sell(size=POSITION_SIZE)
                    self.entry_type = 'SHORT'
                    self.entry_price = current_price
                    self.entry_time = current_time
                    self.entry_vwap = vwap
                    self.entry_upper_band = upper_band
                    self.entry_lower_band = lower_band
                    self.entry_rsi = rsi
                    self.entry_bar = len(self)
                    self.trade_count += 1
                    self.in_position = True
                    return

# ═════════════════════════════════════════════════════════════════════════════════
# RUN BACKTEST
# ═════════════════════════════════════════════════════════════════════════════════

if backtest_df is not None and len(backtest_df) > 100:
    all_trades = []  # Clear previous trades
    
    print("📋 BACKTEST CONFIGURATION (from Block 2):")
    print(f"   Trading Mode:            {TRADING_MODE}")
    print(f"   VWAP Std Dev:            {VWAP_STD_DEV_MULTIPLIER}")
    print(f"   LONG Distance:           {LONG_DISTANCE_PERCENT}%")
    print(f"   SHORT Distance:          {SHORT_DISTANCE_PERCENT}%")
    print(f"   Position Size:           {POSITION_SIZE} BTC")
    print(f"   Initial Capital:         ${INITIAL_CAPITAL:,}\n")
    
    # Prepare for Backtrader
    df_bt = backtest_df.set_index('timestamp')
    df_bt.index.name = 'datetime'
    
    # Create Cerebro engine
    cerebro = bt.Cerebro()
    data_feed = bt.feeds.PandasData(dataname=df_bt)
    cerebro.adddata(data_feed)
    
    # Add strategy
    cerebro.addstrategy(VWAPMeanReversionStrategy)
    cerebro.broker.setcash(INITIAL_CAPITAL)
    
    # Add analyzers
    cerebro.addanalyzer(bt.analyzers.TradeAnalyzer, _name='trades')
    cerebro.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')
    cerebro.addanalyzer(bt.analyzers.Returns, _name='returns')
    
    print("⏳ Running backtest...\n")
    results = cerebro.run()
    strat = results[0]
    
    final_value = cerebro.broker.getvalue()
    net_profit = final_value - INITIAL_CAPITAL
    roi = (net_profit / INITIAL_CAPITAL) * 100
    
    # Extract trade analytics
    trade_analysis = strat.analyzers.trades.get_analysis()
    total_trades = trade_analysis.get('total', {}).get('total', 0)
    win_trades = trade_analysis.get('won', {}).get('total', 0)
    loss_trades = trade_analysis.get('lost', {}).get('total', 0)
    
    max_dd_data = strat.analyzers.drawdown.get_analysis()
    max_dd = max_dd_data.get('max', {}).get('drawdown', 0)
    
    win_rate = (win_trades / total_trades * 100) if total_trades > 0 else 0
    
    # Profit factor
    profit_factor = 0
    if trade_analysis.get('lost') and trade_analysis.get('lost').get('pnl'):
        losing_pnl = abs(trade_analysis['lost']['pnl']['total'])
        winning_pnl = trade_analysis.get('won', {}).get('pnl', {}).get('total', 0)
        if losing_pnl > 0:
            profit_factor = winning_pnl / losing_pnl
    
    print("╔" + "═"*78 + "╗")
    print("║" + "✅ VWAP MEAN REVERSION BACKTEST RESULTS".center(78) + "║")
    print("╚" + "═"*78 + "╝\n")
    
    print(f"💰 CAPITAL METRICS:")
    print(f"   Starting Capital:      ${INITIAL_CAPITAL:>12,.2f}")
    print(f"   Final Capital:         ${final_value:>12,.2f}")
    print(f"   Net Profit/Loss:       ${net_profit:>12,.2f}")
    print(f"   ROI:                   {roi:>13.2f}%\n")
    
    print(f"📈 TRADE METRICS:")
    print(f"   Total Trades:          {len(all_trades):>15}")
    print(f"   Winning Trades:        {win_trades:>15}")
    print(f"   Losing Trades:         {loss_trades:>15}")
    print(f"   Win Rate:              {win_rate:>14.2f}%")
    print(f"   Profit Factor:         {profit_factor:>14.2f}\n")
    
    print(f"📉 RISK METRICS:")
    print(f"   Max Drawdown:          {max_dd:>14.2f}%\n")
    
    if len(all_trades) > 0:
        print(f"✅ {len(all_trades)} trades recorded for visualization\n")
    else:
        print(f"⚠️  No trades executed - adjust entry parameters\n")
else:
    print("❌ Insufficient data for backtest (need > 100 candles)")


                🚀 RUNNING OPTIMIZED VWAP MEAN REVERSION BACKTEST                

📋 BACKTEST CONFIGURATION (from Block 2):
   Trading Mode:            BOTH
   VWAP Std Dev:            1.8
   LONG Distance:           1%
   SHORT Distance:          1%
   Position Size:           1.0 BTC
   Initial Capital:         $100,000

⏳ Running backtest...

╔══════════════════════════════════════════════════════════════════════════════╗
║                    ✅ VWAP MEAN REVERSION BACKTEST RESULTS                    ║
╚══════════════════════════════════════════════════════════════════════════════╝

💰 CAPITAL METRICS:
   Starting Capital:      $  100,000.00
   Final Capital:         $  100,000.00
   Net Profit/Loss:       $        0.00
   ROI:                            0.00%

📈 TRADE METRICS:
   Total Trades:                        0
   Winning Trades:                      0
   Losing Trades:                       0
   Win Rate:                        0.00%
   Profit Factor:                   0.00



---
## BLOCK 6️⃣ : TRADINGVIEW-STYLE TRADE LOGS

In [6]:
# ═══════════════════════════════════════════════════════════════════════════════
# BLOCK 6: PROFESSIONAL TRADE LOGS WITH RAINBOW CSV & ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*100)
print("📋 PROFESSIONAL TRADE ANALYSIS & LOGS".center(100))
print("="*100 + "\n")

# Initialize global variable
backtest_trade_log = None

# ═════════════════════════════════════════════════════════════════════════════════
# ENHANCED TRADE LOG GENERATION WITH ADVANCED METRICS
# ═════════════════════════════════════════════════════════════════════════════════

def generate_professional_trade_log(trades_list):
    """
    Generate comprehensive trade analysis with professional metrics
    Includes risk/reward ratios, trade duration, price action analysis
    """
    
    if not trades_list or len(trades_list) == 0:
        print("⚠️  NO TRADES EXECUTED IN THIS BACKTEST")
        print("   Suggested Adjustments:")
        print("   - Reduce VWAP threshold (lower distance %)")
        print("   - Adjust RSI levels (RSI_OVERSOLD/OVERBOUGHT)")
        print("   - Increase backtest period")
        print("   - Check data availability\n")
        return None
    
    trades = []
    cumulative_pnl = 0
    cumulative_trades_pnl = 0
    max_profit = 0
    max_loss = 0
    
    for i, trade in enumerate(trades_list):
        cumulative_pnl = cumulative_trades_pnl + trade['pnl']
        trade_duration = trade['duration_bars']
        
        # Price action metrics
        if trade['entry_type'] == 'LONG':
            price_move = ((trade['exit_price'] - trade['entry_price']) / trade['entry_price']) * 100
            vwap_distance_entry = ((trade['entry_price'] - trade['entry_vwap']) / trade['entry_vwap']) * 100
            vwap_distance_exit = ((trade['exit_price'] - trade['exit_vwap']) / trade['exit_vwap']) * 100
            band_distance = ((trade['entry_price'] - trade['entry_lower']) / trade['entry_lower']) * 100
        else:  # SHORT
            price_move = ((trade['entry_price'] - trade['exit_price']) / trade['entry_price']) * 100
            vwap_distance_entry = ((trade['entry_vwap'] - trade['entry_price']) / trade['entry_vwap']) * 100
            vwap_distance_exit = ((trade['exit_vwap'] - trade['exit_price']) / trade['exit_vwap']) * 100
            band_distance = ((trade['entry_upper'] - trade['entry_price']) / trade['entry_upper']) * 100
        
        # Risk/Reward
        if trade['pnl'] > 0:
            max_profit = max(max_profit, trade['pnl'])
        else:
            max_loss = min(max_loss, trade['pnl'])
        
        # Trade quality metrics
        bars_to_profit = trade['duration_bars']
        avg_profit_per_bar = trade['pnl'] / bars_to_profit if bars_to_profit > 0 else 0
        
        entry_time = pd.to_datetime(trade['entry_time'])
        exit_time = pd.to_datetime(trade['exit_time'])
        
        trades.append({
            'Trade #': i + 1,
            'Status': '✓ WIN' if trade['pnl'] > 0 else '✗ LOSS',
            'Type': trade['entry_type'],
            'Entry Time': entry_time.strftime('%Y-%m-%d %H:%M'),
            'Exit Time': exit_time.strftime('%Y-%m-%d %H:%M'),
            'Duration (min)': trade['duration_bars'],
            'Entry Price': round(trade['entry_price'], 2),
            'Entry VWAP': round(trade['entry_vwap'], 2),
            'Entry Distance to VWAP (%)': round(vwap_distance_entry, 3),
            'Distance to Band (%)': round(band_distance, 3),
            'Entry RSI': round(trade['entry_rsi'], 2),
            'Exit Price': round(trade['exit_price'], 2),
            'Exit VWAP': round(trade['exit_vwap'], 2),
            'Exit Distance to VWAP (%)': round(vwap_distance_exit, 3),
            'Exit RSI': round(trade['exit_rsi'], 2),
            'Price Move (%)': round(price_move, 3),
            'Position Size': trade['position_size'],
            'P&L ($)': round(trade['pnl'], 2),
            'P&L (%)': round(trade['pnl_pct'], 3),
            'Profit/Bar ($)': round(avg_profit_per_bar, 4),
            'Exit Reason': trade['exit_reason'],
            'Cumulative P&L ($)': round(cumulative_pnl, 2),
        })
        
        cumulative_trades_pnl += trade['pnl']
    
    return pd.DataFrame(trades)

if backtest_df is not None and len(all_trades) > 0:
    trade_log = generate_professional_trade_log(all_trades)
    
    if trade_log is not None:
        backtest_trade_log = trade_log  # Store globally
        
        print(f"📊 TOTAL TRADES EXECUTED: {len(trade_log)}\n")
        print("="*160)
        print(f"{'#':<3} {'Status':<8} {'Type':<6} {'Entry Time':<16} {'Exit Time':<16} {'Duration':<10} {'Entry $':<12} {'Exit $':<12} {'P&L $':<12} {'P&L %':<10} {'Reason':<15}")
        print("="*160)
        
        for idx, trade in trade_log.iterrows():
            print(f"{trade['Trade #']:<3} {trade['Status']:<8} {trade['Type']:<6} {trade['Entry Time']:<16} {trade['Exit Time']:<16} {trade['Duration (min)']:<10} ${trade['Entry Price']:<11,.2f} ${trade['Exit Price']:<11,.2f} ${trade['P&L ($)']:<11,.2f} {trade['P&L (%)']:<9.2f}% {trade['Exit Reason']:<15}")
        
        print("="*160 + "\n")
        
        # Summary statistics
        total_pnl = trade_log['P&L ($)'].sum()
        winners = (trade_log['P&L ($)'] > 0).sum()
        losers = (trade_log['P&L ($)'] < 0).sum()
        avg_win = trade_log[trade_log['P&L ($)'] > 0]['P&L ($)'].mean() if winners > 0 else 0
        avg_loss = trade_log[trade_log['P&L ($)'] < 0]['P&L ($)'].mean() if losers > 0 else 0
        max_win = trade_log['P&L ($)'].max()
        max_loss_val = trade_log['P&L ($)'].min()
        
        print("📈 SUMMARY STATISTICS:")
        print(f"   Total Trades:           {len(trade_log)}")
        print(f"   Winning Trades:         {winners} ({winners/len(trade_log)*100:.1f}%)")
        print(f"   Losing Trades:          {losers} ({losers/len(trade_log)*100:.1f}%)")
        print(f"   Avg Win:                ${avg_win:,.2f}")
        print(f"   Avg Loss:               ${avg_loss:,.2f}")
        print(f"   Max Win:                ${max_win:,.2f}")
        print(f"   Max Loss:               ${max_loss_val:,.2f}")
        print(f"   Total P&L:              ${total_pnl:,.2f}")
        print(f"   Final Cumulative P&L:   ${trade_log['Cumulative P&L ($)'].iloc[-1]:,.2f}\n")
        
        # Export to CSV with enhanced formatting
        csv_file = 'vwap_trade_analysis.csv'
        trade_log.to_csv(csv_file, index=False)
        print(f"✅ Trade log exported to: {csv_file}")
        print(f"   Columns: {len(trade_log.columns)}")
        print(f"   Include: Trade details, VWAP analysis, RSI levels, P&L metrics\n")
        
        # Create a summary export too
        summary_data = {
            'Metric': [
                'Total Capital',
                'Net Profit/Loss',
                'ROI (%)',
                'Total Trades',
                'Winning Trades',
                'Losing Trades',
                'Win Rate (%)',
                'Avg Win ($)',
                'Avg Loss ($)',
                'Max Win ($)',
                'Max Loss ($)',
                'Profit Factor',
            ],
            'Value': [
                INITIAL_CAPITAL,
                total_pnl,
                (total_pnl / INITIAL_CAPITAL) * 100,
                len(trade_log),
                winners,
                losers,
                (winners / len(trade_log) * 100) if len(trade_log) > 0 else 0,
                avg_win,
                avg_loss,
                max_win,
                max_loss_val,
                abs(avg_win / avg_loss) if avg_loss != 0 else 0,
            ]
        }
        
        summary_df = pd.DataFrame(summary_data)
        summary_file = 'vwap_backtest_summary.csv'
        summary_df.to_csv(summary_file, index=False)
        print(f"✅ Summary exported to: {summary_file}\n")
        
    else:
        print("❌ Could not generate trade log")
        backtest_trade_log = None
else:
    backtest_trade_log = None
    if backtest_df is not None:
        print("⚠️  NO TRADES GENERATED\n")


                                📋 PROFESSIONAL TRADE ANALYSIS & LOGS                                

⚠️  NO TRADES GENERATED



---
## BLOCK 7️⃣ : INTERACTIVE CANDLESTICK CHARTS WITH INDICATORS

In [7]:
# ═══════════════════════════════════════════════════════════════════════════════
# SETUP: Install & Verify Required Libraries for Advanced Charting
# ═══════════════════════════════════════════════════════════════════════════════

import subprocess
import sys

print("📦 Installing/Verifying charting libraries...\n")

# Libraries to install
libraries_to_install = [
    ('plotly', 'plotly'),
    ('kaleido', 'kaleido'),
]

for lib_module, lib_pip in libraries_to_install:
    try:
        __import__(lib_module)
        print(f"✓ {lib_pip} already installed")
    except ImportError:
        print(f"⏳ Installing {lib_pip}...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", lib_pip, "-q"], 
                                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            print(f"✓ {lib_pip} installed successfully")
        except Exception as e:
            print(f"⚠️  {lib_pip} installation skipped: {str(e)[:50]}")

print("\n✅ All essential libraries ready!\n")

# Pre-declare global variable
backtest_trade_log = None
print("✓ Global variables initialized\n")

📦 Installing/Verifying charting libraries...

✓ plotly already installed
✓ kaleido already installed

✅ All essential libraries ready!

✓ Global variables initialized



In [8]:
# ═══════════════════════════════════════════════════════════════════════════════
# BLOCK 7: PROFESSIONAL INTERACTIVE CHARTS WITH ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════════

import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("\n" + "="*80)
print("📊 GENERATING PROFESSIONAL INTERACTIVE CHART".center(80))
print("="*80 + "\n")

# Check if backtest data exists
if backtest_df is None or len(backtest_df) == 0:
    print("❌ No data available for charting")
    print("   Please run Block 4 first to fetch data\n")
else:
    print("✓ Data available, preparing chart...\n")
    
    # Prepare chart data
    chart_df = backtest_df.copy()
    chart_df = calculate_vwap_bands(chart_df, VWAP_ANCHOR, VWAP_STD_DEV_MULTIPLIER)
    chart_df = calculate_rsi(chart_df, RSI_PERIOD)
    
    # Use last 2000 candles for optimal performance
    chart_data = chart_df.tail(2000).reset_index(drop=True)
    
    print(f"📈 Creating chart with {len(chart_data)} candles...\n")
    
    # ═════════════════════════════════════════════════════════════════════════════════
    # CREATE MULTI-SUBPLOT FIGURE
    # ═════════════════════════════════════════════════════════════════════════════════
    
    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.12,
        row_heights=[0.6, 0.2, 0.2],
        specs=[[{"secondary_y": False}], 
               [{"secondary_y": False}], 
               [{"secondary_y": False}]]
    )
    
    # ═════════════════════════════════════════════════════════════════════════════════
    # ROW 1: CANDLESTICK CHART
    # ═════════════════════════════════════════════════════════════════════════════════
    
    fig.add_trace(
        go.Candlestick(
            x=chart_data['timestamp'],
            open=chart_data['open'],
            high=chart_data['high'],
            low=chart_data['low'],
            close=chart_data['close'],
            name='OHLC',
            showlegend=True,
        ),
        row=1, col=1
    )
    
    # Add VWAP line
    fig.add_trace(
        go.Scatter(
            x=chart_data['timestamp'],
            y=chart_data['vwap'],
            name='VWAP',
            line=dict(color='blue', width=2),
            showlegend=True,
        ),
        row=1, col=1
    )
    
    # Add Upper Band
    fig.add_trace(
        go.Scatter(
            x=chart_data['timestamp'],
            y=chart_data['upper_band'],
            name='Upper Band',
            line=dict(color='red', width=1, dash='dash'),
            showlegend=True,
        ),
        row=1, col=1
    )
    
    # Add Lower Band with fill
    fig.add_trace(
        go.Scatter(
            x=chart_data['timestamp'],
            y=chart_data['lower_band'],
            name='Lower Band',
            line=dict(color='green', width=1, dash='dash'),
            fill='tonexty',
            fillcolor='rgba(128, 128, 128, 0.1)',
            showlegend=True,
        ),
        row=1, col=1
    )
    
    # ═════════════════════════════════════════════════════════════════════════════════
    # ADD TRADE MARKERS IF AVAILABLE
    # ═════════════════════════════════════════════════════════════════════════════════
    
    if backtest_trade_log is not None and len(backtest_trade_log) > 0:
        print(f"📍 Adding {len(backtest_trade_log)} trade markers...\n")
        
        # Buy signals
        buy_trades = backtest_trade_log[backtest_trade_log['Type'] == 'LONG']
        if len(buy_trades) > 0:
            buy_times = pd.to_datetime(buy_trades['Entry Time'])
            buy_prices = buy_trades['Entry Price']
            fig.add_trace(
                go.Scatter(
                    x=buy_times,
                    y=buy_prices,
                    mode='markers',
                    name='LONG Entry',
                    marker=dict(size=12, color='green', symbol='triangle-up', 
                               line=dict(color='white', width=2)),
                    showlegend=True,
                ),
                row=1, col=1
            )
        
        # Sell signals
        sell_trades = backtest_trade_log[backtest_trade_log['Type'] == 'SHORT']
        if len(sell_trades) > 0:
            sell_times = pd.to_datetime(sell_trades['Entry Time'])
            sell_prices = sell_trades['Entry Price']
            fig.add_trace(
                go.Scatter(
                    x=sell_times,
                    y=sell_prices,
                    mode='markers',
                    name='SHORT Entry',
                    marker=dict(size=12, color='red', symbol='triangle-down', 
                               line=dict(color='white', width=2)),
                    showlegend=True,
                ),
                row=1, col=1
            )
    
    # ═════════════════════════════════════════════════════════════════════════════════
    # ROW 2: VOLUME CHART
    # ═════════════════════════════════════════════════════════════════════════════════
    
    colors = ['red' if chart_data['close'].iloc[i] < chart_data['open'].iloc[i] else 'green' 
              for i in range(len(chart_data))]
    
    fig.add_trace(
        go.Bar(
            x=chart_data['timestamp'],
            y=chart_data['volume'],
            name='Volume',
            marker=dict(color=colors),
            showlegend=True,
        ),
        row=2, col=1
    )
    
    # ═════════════════════════════════════════════════════════════════════════════════
    # ROW 3: RSI CHART
    # ═════════════════════════════════════════════════════════════════════════════════
    
    fig.add_trace(
        go.Scatter(
            x=chart_data['timestamp'],
            y=chart_data['rsi'],
            name='RSI(14)',
            line=dict(color='orange', width=2),
            showlegend=True,
        ),
        row=3, col=1
    )
    
    # Add RSI reference lines
    fig.add_hline(y=70, line_dash="dash", line_color="red", 
                 annotation_text="Overbought (70)", row=3, col=1)
    fig.add_hline(y=30, line_dash="dash", line_color="green", 
                 annotation_text="Oversold (30)", row=3, col=1)
    
    # ═════════════════════════════════════════════════════════════════════════════════
    # UPDATE LAYOUT
    # ═════════════════════════════════════════════════════════════════════════════════
    
    fig.update_layout(
        title=f'VWAP Mean Reversion Strategy - {SYMBOL} {TIMEFRAME}',
        template='plotly_dark',
        height=1000,
        hovermode='x unified',
        xaxis=dict(
            rangeslider=dict(visible=False),
            type='date',
        ),
        xaxis2=dict(rangeslider=dict(visible=False)),
        xaxis3=dict(rangeslider=dict(visible=False)),
    )
    
    fig.update_yaxes(title_text='Price (USD)', row=1, col=1)
    fig.update_yaxes(title_text='Volume', row=2, col=1)
    fig.update_yaxes(title_text='RSI', row=3, col=1)
    
    # Save chart
    chart_file = 'vwap_trading_chart.html'
    fig.write_html(chart_file)
    
    print("╔" + "═"*78 + "╗")
    print("║" + "✅ INTERACTIVE CHART CREATED".center(78) + "║")
    print("╚" + "═"*78 + "╝")
    print(f"\n📊 CHART FEATURES:")
    print(f"   ✓ Candlestick OHLC chart")
    print(f"   ✓ VWAP line (Blue)")
    print(f"   ✓ Upper/Lower confidence bands")
    print(f"   ✓ Volume histogram")
    print(f"   ✓ RSI(14) indicator")
    if backtest_trade_log is not None:
        print(f"   ✓ Trade entry/exit markers ({len(backtest_trade_log)} trades)")
    print(f"   ✓ Fully interactive (zoom, pan, hover)")
    print(f"   ✓ Time range selector")
    
    print(f"\n📈 Chart Statistics:")
    print(f"   Total Candles: {len(chart_data):,}")
    print(f"   Time Range: {chart_data['timestamp'].min().strftime('%Y-%m-%d %H:%M')} to {chart_data['timestamp'].max().strftime('%Y-%m-%d %H:%M')}")
    
    print(f"\n💾 Chart saved to: {chart_file}")
    print(f"\n✅ Open in browser: {chart_file}\n")
    
    # Try to open in browser
    try:
        import webbrowser
        import os as os_module
        chart_path = os_module.path.abspath(chart_file)
        webbrowser.open('file://' + chart_path)
        print("🌐 Opening chart in default browser...\n")
    except:
        print("📍 Please open the HTML file manually in your browser\n")


                  📊 GENERATING PROFESSIONAL INTERACTIVE CHART                   

✓ Data available, preparing chart...

📈 Creating chart with 2000 candles...

╔══════════════════════════════════════════════════════════════════════════════╗
║                         ✅ INTERACTIVE CHART CREATED                          ║
╚══════════════════════════════════════════════════════════════════════════════╝

📊 CHART FEATURES:
   ✓ Candlestick OHLC chart
   ✓ VWAP line (Blue)
   ✓ Upper/Lower confidence bands
   ✓ Volume histogram
   ✓ RSI(14) indicator
   ✓ Fully interactive (zoom, pan, hover)
   ✓ Time range selector

📈 Chart Statistics:
   Total Candles: 2,000
   Time Range: 2026-03-25 01:42 to 2026-03-26 11:01

💾 Chart saved to: vwap_trading_chart.html

✅ Open in browser: vwap_trading_chart.html

🌐 Opening chart in default browser...

